# Week 21 ERP Image Export for Label Studio

This notebook exports the unique Week-21 data sources selected from the ERP image screening screenshots. The duplicate `eye_eeg_reading_fixations` screenshot is intentionally represented only once. The already labelled Week-20 training source `notebooks/model_test/real_data_sets/fixations_dataset` is explicitly excluded and guarded against in the helper module.

Export conventions:
- one folder per data source under `notebooks/week_21`
- rendered PNG images plus `manifest.csv`, `tasks_*.json`, `source_config.json`, `source_reference.json`, `README.md`, and `labeling_interface.xml`
- `source_reference.json` stores the source URL/script/docs fields from the Week-19 dataset metadata once per data source for thesis reference checks
- ERP matrices are sorted, z-scored per time point, and Gaussian-smoothed
- no `64x64` matrix resize is applied
- sort variables are restricted to experiment/behaviour variables that can plausibly expose ERP image patterns from the thesis section `ERP Image Patterns`

The first executable export below writes only `10` tasks per data source into the test folder. The full `1000`-per-source export stays commented until the test output has been checked.


## Label Studio Local Storage Notes

For local-file based imports, Label Studio needs local-file serving enabled and a document root. This export assumes:

```bash
export LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED=true
export LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT=/home/benjamin/Dokumente/BA2/notebooks/week_21
```

The task JSON files use `/data/local-files/?d=...` image URLs relative to that document root. In Label Studio, import each generated `tasks_*.json`, or add the data-source folder as a Local Files source storage and use import method `Tasks`.

Docs checked: [Local storage](https://labelstud.io/guide/storage_local) and [cloud/local storage setup](https://labelstud.io/guide/storage).

In [ ]:
import Pkg

ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"
ENV["JULIA_NUM_PRECOMPILE_TASKS"] = "1"

function find_repo_root(start_dir = pwd())
    candidates = unique(normpath.([
        start_dir,
        joinpath(start_dir, ".."),
        joinpath(start_dir, "..", ".."),
        joinpath(start_dir, "..", "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
    end
    error("Could not locate repository root from pwd=$(pwd()).")
end

repo_root = find_repo_root()
week21_dir = joinpath(repo_root, "notebooks", "week_21")
Pkg.activate(joinpath(repo_root, "notebooks", "model_test"))

using CSV
using DataFrames

include(joinpath(week21_dir, "labelstudio_erp_export_helpers.jl"))
using .Week21LabelStudioERPExport

println("Week-21 notebook dir: ", NOTEBOOK_DIR)
println("Test export root: ", TEST_EXPORT_ROOT)
println("Full export root: ", FULL_EXPORT_ROOT)

## Source Plan

This table checks the locally available Week-19 bundles, selected sort variables, channel counts, and the variant capacity needed for the later `1000`-image export.

In [ ]:
overview_df = source_overview_df(; target_count = 1000)
overview_df

## Test Export

Run this first. It writes `10` Label Studio tasks per data source into `notebooks/week_21/labelstudio_export_test`.

In [ ]:
test_export = export_labelstudio_test()
test_export.summary_df

In [ ]:
first_manifest = test_export.summary_df.manifest_path[1]
first(CSV.read(first_manifest, DataFrame), 10)

## Full Export Gate

Leave this commented until the test directory has been reviewed. The full run writes `1000` tasks per unique data source into `notebooks/week_21/labelstudio_export_full`.

In [ ]:
# full_export = export_labelstudio_full()
# full_export.summary_df